In [1]:
!git clone https://github.com/ageron/handson-ml2.git

fatal: destination path 'handson-ml2' already exists and is not an empty directory.


In [2]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt

In [ ]:
path_csv = "/content/handson-ml2/datasets/housing/housing.csv"
housing = pd.read_csv(path_csv)
housing.head()

In [4]:
housing.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   longitude           20640 non-null  float64
 1   latitude            20640 non-null  float64
 2   housing_median_age  20640 non-null  float64
 3   total_rooms         20640 non-null  float64
 4   total_bedrooms      20433 non-null  float64
 5   population          20640 non-null  float64
 6   households          20640 non-null  float64
 7   median_income       20640 non-null  float64
 8   median_house_value  20640 non-null  float64
 9   ocean_proximity     20640 non-null  object 
dtypes: float64(9), object(1)
memory usage: 1.6+ MB


In [5]:
housing["ocean_proximity"].value_counts()

,count
ocean_proximity,
<1H OCEAN,9136
INLAND,6551
NEAR OCEAN,2658
NEAR BAY,2290
ISLAND,5


In [ ]:
housing.describe()

In [ ]:
housing.hist(bins=50, figsize=(15,12))
plt.show()

In [8]:
from sklearn.model_selection import train_test_split, StratifiedShuffleSplit
train_set, test_set = train_test_split(housing, test_size=0.2, random_state=42)

In [ ]:
housing["income_cat"] = pd.cut(housing["median_income"],
                               bins=[0., 1.5, 3.0, 4.5, 6., np.inf],
                               labels=[1, 2, 3, 4, 5])

housing["income_cat"].hist()

In [10]:
split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

for train_index, test_index in split.split(housing, housing["income_cat"]):
  strat_train_set = housing.loc[train_index]
  strat_test_set = housing.loc[test_index]

In [11]:
strat_test_set["income_cat"].value_counts() / len(strat_test_set)

,count
income_cat,
3,0.350533
2,0.318798
4,0.176357
5,0.114341
1,0.039971


In [12]:
for set_ in (strat_train_set, strat_test_set):
  set_.drop("income_cat", axis=1, inplace=True)

In [ ]:
housing = strat_train_set.copy()
housing.plot(kind="scatter", x="longitude", y="latitude",
             alpha=0.4,
             s=housing["population"]/100, label="population",
             figsize=(10,7),
             c="median_house_value", cmap=plt.get_cmap("jet"),
             colorbar=True)
plt.legend()
plt.show()

In [14]:
corr_matrix = housing.drop("ocean_proximity", axis=1).corr()
corr_matrix["median_house_value"].sort_values(ascending=False)

,median_house_value
median_house_value,1.000000
median_income,0.687151
total_rooms,0.135140
housing_median_age,0.114146
households,0.064590
total_bedrooms,0.047781
population,-0.026882
longitude,-0.047466
latitude,-0.142673


In [ ]:
from pandas.plotting import scatter_matrix
attributes = ["median_house_value", "median_income", "total_rooms", "housing_median_age"]
scatter_matrix(housing[attributes], figsize=(12,8));

In [16]:
housing["rooms_per_household"] = housing["total_rooms"]/housing["households"]
housing["bedrooms_per_room"] = housing["total_bedrooms"]/housing["total_rooms"]
housing["population_per_household"] = housing["population"]/housing["households"]

In [17]:
corr_matrix = housing.drop("ocean_proximity", axis=1).corr()
corr_matrix["median_house_value"].sort_values(ascending=False)

,median_house_value
median_house_value,1.000000
median_income,0.687151
rooms_per_household,0.146255
total_rooms,0.135140
housing_median_age,0.114146
households,0.064590
total_bedrooms,0.047781
population_per_household,-0.021991
population,-0.026882
longitude,-0.047466


In [18]:
housing = train_set.drop(["median_house_value"], axis=1)
housing_labels = strat_train_set["median_house_value"].copy()

In [19]:
median_total_bedrooms = housing["total_bedrooms"].median()
housing["total_bedrooms"].fillna(median_total_bedrooms, inplace=True)

/tmp/ipython-input-3083990056.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  housing["total_bedrooms"].fillna(median_total_bedrooms, inplace=True)


In [20]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")
housing_num = housing.drop("ocean_proximity", axis=1)
imputer.fit(housing_num)
imputer.statistics_

array([-118.51  ,   34.26  ,   29.    , 2129.    ,  437.    , 1167.    ,
        410.    ,    3.5458])

In [ ]:
X = imputer.transform(housing_num)
housing_tr = pd.DataFrame(X,
                           columns=housing_num.columns,
                           index=housing_num.index)
housing_tr

In [22]:
housing_cat = housing[["ocean_proximity"]]
housing_cat.head()

,ocean_proximity
14196,NEAR OCEAN
8267,NEAR OCEAN
17445,NEAR OCEAN
14265,NEAR OCEAN
2271,INLAND


In [23]:
from sklearn.preprocessing import OrdinalEncoder
ordinal_encoder = OrdinalEncoder()
housing_cat_encoded = ordinal_encoder.fit_transform(housing_cat)
housing_cat_encoded[:10]

array([[4.],
       [4.],
       [4.],
       [4.],
       [1.],
       [0.],
       [0.],
       [3.],
       [0.],
       [0.]])

In [24]:
ordinal_encoder.categories_

[array(['<1H OCEAN', 'INLAND', 'ISLAND', 'NEAR BAY', 'NEAR OCEAN'],
       dtype=object)]

In [25]:
from sklearn.preprocessing import OneHotEncoder
cat_encoder = OneHotEncoder()
housing_cat_1hot = cat_encoder.fit_transform(housing_cat)
housing_cat_1hot.toarray()

array([[0., 0., 0., 0., 1.],
       [0., 0., 0., 0., 1.],
       [0., 0., 0., 0., 1.],
       ...,
       [1., 0., 0., 0., 0.],
       [1., 0., 0., 0., 0.],
       [0., 0., 0., 1., 0.]])

In [26]:
from sklearn.base import BaseEstimator, TransformerMixin

rooms_ix, bedrooms_ix, population_ix, households_ix = 3, 4, 5, 6

In [27]:
class CombinedAttributesAdder(BaseEstimator, TransformerMixin):
  def __init__(self, add_bedrooms_per_room=True):
    self.add_bedrooms_per_room = add_bedrooms_per_room

  def fit(self, X, y=None):
    return self

  def transform(self, X, y=None):
    rooms_per_household = X[:, rooms_ix] / X[:, households_ix]
    population_per_household = X[:, population_ix] / X[:, households_ix]
    if self.add_bedrooms_per_room:
      bedrooms_per_room = X[:, bedrooms_ix] / X[:, rooms_ix]
      return np.c_[X, rooms_per_household, population_per_household, bedrooms_per_room]
    else:
      return np.c_[X, rooms_per_household, population_per_household]

In [28]:
attr_adder = CombinedAttributesAdder(add_bedrooms_per_room=False)
housing_extra_attribs = attr_adder.transform(housing.values)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

num_pipeline = Pipeline([
                         ('imputer', SimpleImputer(strategy="median")),
                         ('attribs_adder', CombinedAttributesAdder()),
                         ('std_scaler', StandardScaler())
])

housing_num_tr = num_pipeline.fit_transform(housing_num)
housing_tr

In [30]:
from sklearn.compose import ColumnTransformer

num_attribs = list(housing_num)
cat_attribs = ["ocean_proximity"]

full_pipeline = ColumnTransformer([
                                   ("num", num_pipeline, num_attribs),
                                   ("cat", OneHotEncoder(), cat_attribs)
])

housing_prepared = full_pipeline.fit_transform(housing)

In [ ]:
from sklearn.linear_model import LinearRegression

lin_reg = LinearRegression()
lin_reg.fit(housing_prepared, housing_labels)

In [32]:
some_data = housing.iloc[:5]
some_labels = housing_labels.iloc[:5]
some_data_prepared = full_pipeline.transform(some_data)
print("Predictions:", lin_reg.predict(some_data_prepared))
print("Labels:", list(some_labels))

Predictions: [210957.96212735 202512.82323823 206006.47038839 208399.48749327
 206360.47621263]
Labels: [72100.0, 279600.0, 82700.0, 112500.0, 238300.0]


In [33]:
from sklearn.metrics import mean_squared_error

housing_predictions = lin_reg.predict(housing_prepared)
lin_mse = mean_squared_error(housing_labels, housing_predictions)
lin_rmse = np.sqrt(lin_mse)
lin_rmse

np.float64(115622.01223967873)

In [34]:
from sklearn.tree import DecisionTreeRegressor

tree_reg = DecisionTreeRegressor()
tree_reg.fit(housing_prepared, housing_labels)

housing_predictions = tree_reg.predict(housing_prepared)
tree_mse = mean_squared_error(housing_labels, housing_predictions)
tree_rmse = np.sqrt(tree_mse)
tree_rmse

np.float64(0.0)

In [35]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(tree_reg, housing_prepared, housing_labels,
                         scoring="neg_mean_squared_error", cv=10)
tree_rmse_scores = np.sqrt(-scores)

In [36]:
def display_scores(scores):
  print("Scores:", scores)
  print("Mean:", scores.mean())
  print("Standard deviation:", scores.std())

display_scores(tree_rmse_scores)

Scores: [166520.8177501  171150.09130471 165295.19577505 165865.15096062
 167552.48759366 171683.32054921 169159.72859738 166832.2354967
 169362.04353913 165758.19152427]
Mean: 167917.92630908362
Standard deviation: 2174.2555182491355


In [37]:
lin_scores = cross_val_score(lin_reg, housing_prepared, housing_labels,
                             scoring="neg_mean_squared_error", cv=10)
lin_rmse_scores = np.sqrt(-lin_scores)
display_scores(lin_rmse_scores)

Scores: [114502.5729195  116090.52153142 113020.72759114 117088.48240497
 112943.20286469 118854.60266072 116572.30972232 115440.51145472
 116920.86254444 116156.01068697]
Mean: 115758.98043808764
Standard deviation: 1753.1712362674643


In [38]:
from sklearn.ensemble import RandomForestRegressor

forest_reg = RandomForestRegressor()
forest_reg.fit(housing_prepared, housing_labels)

housing_predictions = forest_reg.predict(housing_prepared)
forest_mse = mean_squared_error(housing_labels, housing_predictions)

forest_mse = np.sqrt(forest_mse)
display_scores(forest_mse)

Scores: 44350.94431438023
Mean: 44350.94431438023
Standard deviation: 0.0


In [39]:
import joblib

joblib.dump(forest_reg, "forest_reg.pkl")

['forest_reg.pkl']

In [40]:
from sklearn.model_selection import GridSearchCV

param_grid = [
              {'n_estimators': [3, 10, 30],
               "max_features": [2, 4, 6, 8]},
              {'bootstrap': [False],
               'n_estimators': [3, 10],
               'max_features': [2, 3, 4]}
]

forest_rag = RandomForestRegressor()

grid_search = GridSearchCV(forest_rag, param_grid, cv=5,
                           scoring="neg_mean_squared_error",
                           return_train_score=True)

In [ ]:
grid_search.fit(housing_prepared, housing_labels)
grid_search.best_estimator_

In [42]:
grid_search.best_params_

{'max_features': 2, 'n_estimators': 30}

In [43]:
cv_scores = grid_search.cv_results_
for mean_score, params in zip(cv_scores["mean_test_score"], cv_scores["params"]):
  print(np.sqrt(-mean_score), params)

134733.0168580788 {'max_features': 2, 'n_estimators': 3}
123385.7822057736 {'max_features': 2, 'n_estimators': 10}
119228.36522063437 {'max_features': 2, 'n_estimators': 30}
135103.89120492677 {'max_features': 4, 'n_estimators': 3}
123252.5172187388 {'max_features': 4, 'n_estimators': 10}
119488.2102078784 {'max_features': 4, 'n_estimators': 30}
136343.31663496382 {'max_features': 6, 'n_estimators': 3}
123663.10923558501 {'max_features': 6, 'n_estimators': 10}
119392.90065545125 {'max_features': 6, 'n_estimators': 30}
136305.10597317226 {'max_features': 8, 'n_estimators': 3}
123790.39400084752 {'max_features': 8, 'n_estimators': 10}
119521.0771312917 {'max_features': 8, 'n_estimators': 30}
134680.0138777119 {'bootstrap': False, 'max_features': 2, 'n_estimators': 3}
123804.0014438331 {'bootstrap': False, 'max_features': 2, 'n_estimators': 10}
136233.64155983366 {'bootstrap': False, 'max_features': 3, 'n_estimators': 3}
124206.97366604862 {'bootstrap': False, 'max_features': 3, 'n_estima

In [44]:
features_importances = grid_search.best_estimator_.feature_importances_
features_importances

array([0.08534458, 0.08237579, 0.07534514, 0.09041177, 0.08932612,
       0.08966771, 0.08903227, 0.09501669, 0.09452193, 0.09385019,
       0.09472482, 0.00637609, 0.00618981, 0.00011005, 0.00296548,
       0.00474156])

In [45]:
extra_attribs = ["rooms_per_hhold", "pop_per_hhold", "bedrooms_per_room"]
cat_encoder = full_pipeline.named_transformers_["cat"]
cat_one_hot_attribs = list(cat_encoder.categories_[0])
attributes = num_attribs + extra_attribs + cat_one_hot_attribs
sorted(zip(features_importances, attributes), reverse=True)

[(np.float64(0.09501668617152677), 'median_income'),
 (np.float64(0.09472481983638548), 'bedrooms_per_room'),
 (np.float64(0.094521927611024), 'rooms_per_hhold'),
 (np.float64(0.09385018821869424), 'pop_per_hhold'),
 (np.float64(0.09041176977094274), 'total_rooms'),
 (np.float64(0.08966771114184956), 'population'),
 (np.float64(0.08932612421116008), 'total_bedrooms'),
 (np.float64(0.08903227377116418), 'households'),
 (np.float64(0.08534458365920035), 'longitude'),
 (np.float64(0.08237578945065342), 'latitude'),
 (np.float64(0.0753451376647668), 'housing_median_age'),
 (np.float64(0.006376091167401894), '<1H OCEAN'),
 (np.float64(0.006189809022480564), 'INLAND'),
 (np.float64(0.004741558161024364), 'NEAR OCEAN'),
 (np.float64(0.0029654848362904943), 'NEAR BAY'),
 (np.float64(0.00011004530543517381), 'ISLAND')]

In [47]:
final_model = grid_search.best_estimator_

X_test = strat_test_set.drop("median_house_value", axis=1)
y_test = strat_test_set["median_house_value"].copy()

X_test_prepared = full_pipeline.transform(X_test)
final_predictions = final_model.predict(X_test_prepared)

final_mse = mean_squared_error(y_test, final_predictions)
final_rmse = np.sqrt(final_mse)
final_rmse

np.float64(134327.87068493257)

In [48]:
from scipy import stats

confidence = 0.95
squared_errors = (final_predictions - y_test) ** 2
np.sqrt(stats.t.interval(confidence, len(squared_errors) - 1,
                         loc=squared_errors.mean(),
                         scale=stats.sem(squared_errors)))

array([131264.33344462, 137323.0805467 ])